In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
import tarfile
import matplotlib.pyplot as plt
plt.rc('font' , size = 14)
plt.rc('axes' , labelsize = 10 , titlesize = 14)
plt.rc('legend' , fontsize = 14)
plt.rc('xtick' , labelsize = 10)
plt.rc('ytick' , labelsize = 10)

In [4]:
from pathlib import Path
IMAGES_PATH = Path() / "images"/"classification"
IMAGES_PATH.mkdir(parents=True , exist_ok= True)

def save_fig(fig_id , tight_layout= True, fig_extension = "png" , resolution = 300):
    path = IMAGES_PATH/f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path , format=fig_extension , dpi = resolution)
    


In [5]:
import urllib.request
import tarfile
def fetch_spam_data():
    spam_root = "http://spamassassin.apache.org/old/publiccorpus/"
    ham_url = spam_root + "20030228_easy_ham.tar.bz2"
    spam_url = spam_root + "20030228_spam.tar.bz2"


    spam_path  = Path() / "datasets" / "spam"
    spam_path.mkdir(parents=True , exist_ok=True)

    for dir_name , tar_name , url in (("easy_ham" , "ham" , ham_url) , ("spam" , "spam" , spam_url)):
        if not (spam_path / dir_name).is_dir():
                
            path = (spam_path / tar_name).with_suffix(".tar.bz2")
            print("Downloading" , path)
            urllib.request.urlretrieve(url,path)
            tar_bz2_file = tarfile.open(path , "r:bz2")
            tar_bz2_file.extractall(path = spam_path)
            tar_bz2_file.close()
    return [spam_path/dir_name for dir_name in("easy_ham" , "spam")]



In [6]:
ham_dir , spam_dir = fetch_spam_data()

In [7]:
print(type(ham_dir))
print(spam_dir)


<class 'pathlib.WindowsPath'>
datasets\spam\spam


In [8]:
ham_filenames = [f for f in sorted(ham_dir.iterdir()) if len(f.name)>20]
spam_filenames = [f for f in sorted(spam_dir.iterdir()) if len(f.name) > 20]


In [9]:
print(len(ham_filenames))
print(len(spam_filenames))

2500
500


In [10]:
import email 
import email.policy 
def parse_email(filepath):
    with open(filepath , "rb") as f:
        return email.parser.BytesParser(policy = email.policy.default).parse(f)


In [11]:
ham_eamils = [parse_email(filepath) for filepath in ham_filenames] 
spam_eamils = [parse_email(filepath) for filepath in spam_filenames] 

In [12]:
print(ham_eamils[1].get_content().strip())

Martin A posted:
Tassos Papadopoulos, the Greek sculptor behind the plan, judged that the
 limestone of Mount Kerdylio, 70 miles east of Salonika and not far from the
 Mount Athos monastic community, was ideal for the patriotic sculpture. 
 
 As well as Alexander's granite features, 240 ft high and 170 ft wide, a
 museum, a restored amphitheatre and car park for admiring crowds are
planned
---------------------
So is this mountain limestone or granite?
If it's limestone, it'll weather pretty fast.

------------------------ Yahoo! Groups Sponsor ---------------------~-->
4 DVDs Free +s&p Join Now
http://us.click.yahoo.com/pt6YBB/NXiEAA/mG3HAA/7gSolB/TM
---------------------------------------------------------------------~->

To unsubscribe from this group, send an email to:
forteana-unsubscribe@egroups.com

 

Your use of Yahoo! Groups is subject to http://docs.yahoo.com/info/terms/


In [13]:
print(spam_eamils[1].get_content().strip())

1) Fight The Risk of Cancer!
http://www.adclick.ws/p.cfm?o=315&s=pk007

2) Slim Down - Guaranteed to lose 10-12 lbs in 30 days
http://www.adclick.ws/p.cfm?o=249&s=pk007

3) Get the Child Support You Deserve - Free Legal Advice
http://www.adclick.ws/p.cfm?o=245&s=pk002

4) Join the Web's Fastest Growing Singles Community
http://www.adclick.ws/p.cfm?o=259&s=pk007

5) Start Your Private Photo Album Online!
http://www.adclick.ws/p.cfm?o=283&s=pk007

Have a Wonderful Day,
Offer Manager
PrizeMama













If you wish to leave this list please use the link below.
http://www.qves.com/trim/?ilug@linux.ie%7C17%7C114258


-- 
Irish Linux Users' Group: ilug@linux.ie
http://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.
List maintainer: listmaster@linux.ie


In [14]:
def get_email_structure(email):
    if isinstance(email, str):
        return email
    payload = email.get_payload()
    if isinstance(payload , list):
        return ",".join([get_email_structure(sub_email) for sub_email in payload])
    else:
        return email.get_content_type()
    

In [15]:
from collections import Counter 
def structures_counter(emails):
    structures = Counter()
    for  email in emails:
        structure = get_email_structure(email)
        structures[structure] += 1
    return structures




In [16]:
structures_counter(ham_eamils).most_common()


[('text/plain', 2411),
 ('text/plain,application/pgp-signature', 66),
 ('text/plain,text/html', 8),
 ('text/plain,text/plain', 5),
 ('text/plain,application/octet-stream', 2),
 ('text/plain,text/enriched', 1),
 ('text/plain,application/ms-tnef,text/plain', 1),
 ('text/plain,text/plain,text/plain,application/pgp-signature', 1),
 ('text/plain,video/mng', 1),
 ('text/plain,application/x-pkcs7-signature', 1),
 ('text/plain,text/plain,text/plain,text/rfc822-headers', 1),
 ('text/plain,text/plain,text/plain,text/plain,application/x-pkcs7-signature',
  1),
 ('text/plain,application/x-java-applet', 1)]

In [17]:
structures_counter(spam_eamils).most_common()

[('text/plain', 237),
 ('text/html', 208),
 ('text/plain,text/html', 45),
 ('text/plain,image/jpeg', 3),
 ('text/html,application/octet-stream', 2),
 ('text/plain,application/octet-stream', 1),
 ('text/html,text/plain', 1),
 ('text/html,application/octet-stream,image/jpeg', 1),
 ('text/plain,text/html,image/gif', 1),
 ('multipart/alternative', 1)]

In [18]:
for header , value in spam_eamils[0].items():
    print(header ,":" , value)
    

Return-Path : <12a1mailbot1@web.de>
Delivered-To : zzzz@localhost.spamassassin.taint.org
Received : from localhost (localhost [127.0.0.1])	by phobos.labs.spamassassin.taint.org (Postfix) with ESMTP id 136B943C32	for <zzzz@localhost>; Thu, 22 Aug 2002 08:17:21 -0400 (EDT)
Received : from mail.webnote.net [193.120.211.219]	by localhost with POP3 (fetchmail-5.9.0)	for zzzz@localhost (single-drop); Thu, 22 Aug 2002 13:17:21 +0100 (IST)
Received : from dd_it7 ([210.97.77.167])	by webnote.net (8.9.3/8.9.3) with ESMTP id NAA04623	for <zzzz@spamassassin.taint.org>; Thu, 22 Aug 2002 13:09:41 +0100
From : 12a1mailbot1@web.de
Received : from r-smtp.korea.com - 203.122.2.197 by dd_it7  with Microsoft SMTPSVC(5.5.1775.675.6);	 Sat, 24 Aug 2002 09:42:10 +0900
To : dcek1a1@netsgo.com
Subject : Life Insurance - Why Pay More?
Date : Wed, 21 Aug 2002 20:31:57 -1600
MIME-Version : 1.0
Message-ID : <0103c1042001882DD_IT7@dd_it7>
Content-Type : text/html; charset="iso-8859-1"
Content-Transfer-Encoding : qu

In [19]:
spam_eamils[0]["Subject"]


'Life Insurance - Why Pay More?'

In [20]:
import numpy as np
from sklearn.model_selection import train_test_split
X = np.array(ham_eamils + spam_eamils , dtype=object)
y = np.array([0]* len(ham_eamils) + [1] * len(spam_eamils))
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 ,random_state= 42 )


In [21]:
from bs4 import BeautifulSoup
import html
def html_to_text(html_content):
    soup = BeautifulSoup(html_content , "html.parser")
    for head in soup(["head"]):
        head.decompose()
    for a_tag in soup("a"):
        a_tag.string = "HYPERLINK"
    text_content = soup.get_text(separator='\n' , strip=True)
    text_content = html.unescape(text_content)
    return text_content


In [22]:
html_spam_emails = [email for email in X_train[y_train == 1] if get_email_structure(email) == "text/html"]
sample_html_spam = html_spam_emails[5]
print(sample_html_spam.get_content().strip()[:1000],"...")



<HR>
<html>
<div bgcolor="#FFFFCC">

  <p align="center"><a
href="http://www.webbasedmailing.com"><img border="0"
src="http://www.webbasedmailing.com/Toners2goLogo.jpg"
width="349" height="96"></a></p>
<p align="center"><font size="6" face="Arial MT
Black"><i>Tremendous Savings</i>
on Toners,&nbsp;</font></p>
<p align="center"><font size="6" face="Arial MT
Black">
Inkjets, FAX, and Thermal Replenishables!!</font></p>
<p><a href="http://www.webbasedmailing.com">Toners 2 Go
</a>is your secret
weapon to lowering your cost for <a
href="http://www.webbasedmailing.com">High Quality,
Low-Cost</a> printer
supplies!&nbsp; We have been in the printer
replenishables business since 1992,
and pride ourselves on rapid response and outstanding
customer service.&nbsp;
What we sell are 100% compatible replacements for
Epson, Canon, Hewlett Packard,
Xerox, Okidata, Brother, and Lexmark; products that
meet and often exceed
original manufacturer's specifications.</p>
<p><i><font size="4">Check out these
p

In [23]:
print(html_to_text(sample_html_spam.get_content())[:])

HYPERLINK
Tremendous Savings
on Toners,
Inkjets, FAX, and Thermal Replenishables!!
HYPERLINK
is your secret
weapon to lowering your cost for
HYPERLINK
printer
supplies!  We have been in the printer
replenishables business since 1992,
and pride ourselves on rapid response and outstanding
customer service. 
What we sell are 100% compatible replacements for
Epson, Canon, Hewlett Packard,
Xerox, Okidata, Brother, and Lexmark; products that
meet and often exceed
original manufacturer's specifications.
Check out these
prices!
Epson Stylus
Color inkjet cartridge
(SO20108):     Epson's Price:
$27.99
Toners2Go price: $9.95!
HP
LaserJet 4 Toner Cartridge
(92298A):           
HP's
Price:
$88.99
Toners2Go
  price: $41.75!
Come visit us on the web to check out our hundreds
of similar bargains at
HYPERLINK
!
request to be excluded by visiting
HYPERLINK
beverley


In [25]:
def email_to_text(email):
    html = None
    for part in email.walk():
        ctype = part.get_content_type()
        if not ctype in ("text/plain" , "text/html"):
            continue
        try:
            content = part.get_content()
        except:
            content = str(part.get_payload())
        if ctype == "text/plain":
            return content
        else:
            html = content
    if html:
        return html_to_text(html)
        

In [26]:
print(email_to_text(sample_html_spam)[:],"....")


HYPERLINK
Tremendous Savings
on Toners,
Inkjets, FAX, and Thermal Replenishables!!
HYPERLINK
is your secret
weapon to lowering your cost for
HYPERLINK
printer
supplies!  We have been in the printer
replenishables business since 1992,
and pride ourselves on rapid response and outstanding
customer service. 
What we sell are 100% compatible replacements for
Epson, Canon, Hewlett Packard,
Xerox, Okidata, Brother, and Lexmark; products that
meet and often exceed
original manufacturer's specifications.
Check out these
prices!
Epson Stylus
Color inkjet cartridge
(SO20108):     Epson's Price:
$27.99
Toners2Go price: $9.95!
HP
LaserJet 4 Toner Cartridge
(92298A):           
HP's
Price:
$88.99
Toners2Go
  price: $41.75!
Come visit us on the web to check out our hundreds
of similar bargains at
HYPERLINK
!
request to be excluded by visiting
HYPERLINK
beverley ....


In [29]:
import nltk
stemmer = nltk.PorterStemmer()
for word in ("Computations","Computation" , "Computing", "Computed" , "Compute","Compulsive"):
    print(word , "=>" , stemmer.stem(word))

for word in ("running","runner" , "runers", "runners"):
    print(word , "=>" , stemmer.stem(word))


Computations => comput
Computation => comput
Computing => comput
Computed => comput
Compute => comput
Compulsive => compuls
running => run
runner => runner
runers => runer
runners => runner


In [28]:
import urlextract
url_extractor = urlextract.URLExtract()
some_text = "will it detect the github.com and https://youtube.com/793-Pq,f?t=3m32ws"
print(url_extractor.find_urls(some_text))


['github.com', 'https://youtube.com/793-Pq,f?t=3m32ws']


In [34]:
from sklearn.base import BaseEstimator , TransformerMixin
import re 
from collections import Counter 

class EmailToWordCounterTransformer(BaseEstimator , TransformerMixin):
    def __init__(self , strip_headers = True , lower_case= True , remove_punctuation = True , replace_urls = True , replace_numbers = True , Stemming = True):
        self.strip_headers = strip_headers
        self.lower_case = lower_case
        self.replace_numbers = replace_numbers
        self.replace_urls = replace_urls
        self.remove_punctuation = remove_punctuation
        self.Stemming = Stemming

    def fit(self , X , y=None):
        return self

    def transform(self , X , y = None):
        X_transformed = []
        for email in X:
            text = email_to_text(email) or ""
            if self.lower_case:
                text = text.lower()
            if self.replace_urls and url_extractor is not None:
                urls = list(set(url_extractor.find_urls(text)))
                urls.sort(key = lambda url: len(url) , reverse=True)
                for url in urls:
                    text = text.replace(url , " URL ")
            if self.replace_numbers:
                text = re.sub(r'\d+(?:\.\d*)?(?:[eE][+-]?\d+)?', 'NUMBER', text)
            if self.remove_punctuation:
                text = re.sub(r'\W+' , ' ', text, flags=re.M)

            word_counts = Counter(text.split())

            if self.Stemming and stemmer is not None:
                stemmed_word_counts = Counter()
                for word , count in word_counts.items():
                    stemmed_word = stemmer.stem(word)
                    stemmed_word_counts[stemmed_word] += count 
                word_counts = stemmed_word_counts
            X_transformed.append(word_counts)
        return np.array(X_transformed)
            

In [35]:
X_few = X_train[:3]
X_few_wordcounts = EmailToWordCounterTransformer().fit_transform(X_few)
X_few_wordcounts

array([Counter({'chuck': 1, 'murcko': 1, 'wrote': 1, 'stuff': 1, 'yawn': 1, 'r': 1}),
       Counter({'the': 11, 'of': 9, 'and': 8, 'all': 3, 'christian': 3, 'to': 3, 'by': 3, 'jefferson': 2, 'i': 2, 'have': 2, 'superstit': 2, 'one': 2, 'on': 2, 'been': 2, 'ha': 2, 'half': 2, 'rogueri': 2, 'teach': 2, 'jesu': 2, 'some': 1, 'interest': 1, 'quot': 1, 'url': 1, 'thoma': 1, 'examin': 1, 'known': 1, 'word': 1, 'do': 1, 'not': 1, 'find': 1, 'in': 1, 'our': 1, 'particular': 1, 'redeem': 1, 'featur': 1, 'they': 1, 'are': 1, 'alik': 1, 'found': 1, 'fabl': 1, 'mytholog': 1, 'million': 1, 'innoc': 1, 'men': 1, 'women': 1, 'children': 1, 'sinc': 1, 'introduct': 1, 'burnt': 1, 'tortur': 1, 'fine': 1, 'imprison': 1, 'what': 1, 'effect': 1, 'thi': 1, 'coercion': 1, 'make': 1, 'world': 1, 'fool': 1, 'other': 1, 'hypocrit': 1, 'support': 1, 'error': 1, 'over': 1, 'earth': 1, 'six': 1, 'histor': 1, 'american': 1, 'john': 1, 'e': 1, 'remsburg': 1, 'letter': 1, 'william': 1, 'short': 1, 'again': 1, 'becom

In [36]:
from scipy.sparse import csr_matrix
class WordCounterToVectorTransformer(BaseEstimator , TransformerMixin):
    def __init__(self , vocabulary_size = 1000):
        self.vocabulary_size = vocabulary_size
    def fit(self , X , y= None):
        total_count = Counter()
        for word_count in X:
            for word , count in word_count.items():
                total_count[word] += min(count , 10)

        most_common = total_count.most_common()[:self.vocabulary_size]
        self.vocabulary_ = {word: index +1 for index , (word , count) in enumerate(most_common)}
        return self 
    
    def transform(self ,X , y = None):
        rows = []
        cols = []
        data = []
        for row , word_count in enumerate(X):
            for word , count in word_count.items():
                rows.append(row)
                cols.append(self.vocabulary_.get(word,0))
                data.append(count)
        return csr_matrix((data , (rows , cols)) , shape=(len(X) , self.vocabulary_size + 1))
                
                 


In [37]:
# Training the spam classifier
from sklearn.pipeline import Pipeline

preprocess_pipeline = Pipeline([
    ("email_to_wordcount", EmailToWordCounterTransformer()),
    ("wordcount_to_vector", WordCounterToVectorTransformer()),
])

X_train_transformed = preprocess_pipeline.fit_transform(X_train)

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

log_clf = LogisticRegression(max_iter=1000, random_state=42)
score = cross_val_score(log_clf, X_train_transformed, y_train, cv=3)
score.mean()

np.float64(0.9858333333333333)

In [40]:
from sklearn.metrics import precision_score, recall_score

X_test_transformed = preprocess_pipeline.transform(X_test)

log_clf = LogisticRegression(max_iter=1000, random_state=42)
log_clf.fit(X_train_transformed, y_train)

y_pred = log_clf.predict(X_test_transformed)

print(f"Precision: {precision_score(y_test, y_pred):.2%}")
print(f"Recall: {recall_score(y_test, y_pred):.2%}")


Precision: 96.88%
Recall: 97.89%
